# AI Agent Prototype — Natural Language Retention Intelligence

**RetentionIQ** | Notebook 06 of 07  
*Prescriptive retention system for 250+ location fitness franchise (Brazil)*

> Previous: [05_optimization.ipynb](./05_optimization.ipynb) — budget optimization under uncertainty  
> Next: [07_monitoring.ipynb](./07_monitoring.ipynb) — drift detection and model monitoring  
> Architecture decisions: [docs/ARCHITECTURE.md](../docs/ARCHITECTURE.md)

## 1. Why Agents, Not Dashboards

Location managers are fitness professionals, not data analysts. They won't read dashboards, they won't interpret p-values, and they definitely won't query SQL. But they will ask: *"Why is churn up at my gym this month?"* and *"What should I do about it?"*

The agent layer translates our entire analytical stack — data queries, survival models, causal estimates, budget optimizer — into natural language interaction. The hard part isn't getting an LLM to generate a good answer. It's building the **guardrails, memory, and evaluation framework** that make it trustworthy.

Three things make this problem harder than a standard chatbot:

1. **Factual grounding:** Every number in the agent's response must trace to a tool call result, not to the LLM's parametric knowledge. The LLM doesn't know our churn rates — the database does.
2. **LGPD compliance:** Brazilian data protection law (LGPD) makes PII handling non-negotiable. Member names, CPFs, emails, and phone numbers must be masked at every boundary. A single leak in production is a legal liability.
3. **Trust calibration:** The agent must know when it doesn't know. A confident wrong answer about retention strategy erodes trust faster than saying "I'm not sure — let me escalate."

This notebook prototypes the agent architecture. Production implementation is in `src/agents/`.

## 2. Architecture Overview

The multi-agent system follows a **router-specialist-writer** pattern. The key design decision: separate agents for analysis vs strategy vs writing. Why not one big agent? Because tool selection becomes unreliable when an agent has 15+ tools. Specialization improves accuracy. The router acts as a lightweight classifier, not a full LLM — keeping latency low.

```
                          ┌─────────────────┐
                          │   User Query     │
                          │  (natural lang)  │
                          └────────┬─────────┘
                                   │
                          ┌────────▼─────────┐
                          │     ROUTER       │
                          │  classifies →    │
                          │  factual /       │
                          │  diagnostic /    │
                          │  prescriptive /  │
                          │  conversational  │
                          └──┬────┬────┬─────┘
                             │    │    │
               ┌─────────────┘    │    └─────────────┐
               │                  │                  │
      ┌────────▼───────┐  ┌──────▼──────┐          │
      │    ANALYST     │  │ (skip for   │          │
      │                │  │  factual)   │          │
      │  Tools:        │  │             │          │
      │  - query_data  │  └─────────────┘          │
      │  - metrics     │                           │
      │  - cohorts     │                           │
      │  - survival    │                           │
      │  - churn_score │                           │
      └────────┬───────┘                           │
               │                                   │
      ┌────────▼───────┐                           │
      │   STRATEGIST   │                           │
      │                │                           │
      │  Tools:        │                           │
      │  - CATE lookup │                           │
      │  - optimizer   │                           │
      │  - simulate    │                           │
      └────────┬───────┘                           │
               │                                   │
               └──────────────┬────────────────────┘
                              │
                     ┌────────▼───────┐
                     │     WRITER     │
                     │                │
                     │  Rules:        │
                     │  - No jargon   │
                     │  - Action items│
                     │  - Contextualized│
                     │    numbers     │
                     └────────┬───────┘
                              │
                     ┌────────▼───────┐
                     │   GUARDRAILS   │
                     │                │
                     │  Layer 1: SQL  │
                     │  Layer 2: DF   │
                     │  Layer 3: Output│
                     │                │
                     │  PII masking   │
                     │  at EVERY      │
                     │  boundary      │
                     └────────┬───────┘
                              │
                     ┌────────▼───────┐
                     │   Response     │
                     │  (manager-     │
                     │   friendly)    │
                     └────────────────┘
```

**Routing logic** (from `src/agents/graph.py`):
- `factual` → Analyst → Writer → END  
- `diagnostic` → Analyst → Strategist → Writer → END  
- `prescriptive` → Analyst → Strategist → Writer → END  
- `conversational` → Writer → END

In [ ]:
import sys
import warnings
from pathlib import Path

import yaml

warnings.filterwarnings("ignore")

# Project root for imports
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Load agent configuration — all parameters externalized to YAML
with open(PROJECT_ROOT / "configs" / "agents.yaml") as f:
    agent_config = yaml.safe_load(f)

print("Agent configuration:")
print(f"  LLM:                     {agent_config['llm']['provider']}/{agent_config['llm']['model']}")
print(f"  Temperature:             {agent_config['llm']['temperature']}")
print(f"  Max tokens:              {agent_config['llm']['max_tokens']}")
print(f"  Embedding model:         {agent_config['embedding']['model']}")
print(f"  Confidence threshold:    {agent_config['guardrails']['confidence_threshold']}")
print(f"  Memory backend:          {agent_config['memory']['backend']}")
print(f"  Conversation window:     {agent_config['memory']['conversation_window']} messages")
print(f"  Long-term retrieval:     top-{agent_config['memory']['long_term_top_k']} (similarity > {agent_config['memory']['similarity_threshold']})")
print(f"  Audit log retention:     {agent_config['audit']['retention_days']} days (LGPD)")
print(f"\nAgents: {list(agent_config['agents'].keys())}")
print(f"Guardrail PII fields: {agent_config['guardrails']['pii_fields']}")

## 3. PII Guardrails Demo

LGPD compliance is non-negotiable. PII is masked at **three layers** of defense:

1. **SQL tool layer:** The SQL generation tool never selects PII columns (`member_name`, `cpf`, `email`, `phone`, `address`). Even if the LLM hallucinates a `SELECT *`, the tool rewrites it.
2. **DataFrame masking:** Every tool that returns member-level data passes through `mask_pii_in_dataframe()` before the data reaches the agent. PII columns are replaced entirely — not partially masked, not hashed.
3. **Output validation:** Before any response reaches the user, `validate_agent_output()` scans for PII patterns (CPF regex, email regex, phone regex). If PII is found, it's masked immediately and a CRITICAL-level log fires.

If PII reaches the user, **all three layers failed** — and we log at CRITICAL level so the team investigates. The implementation is in `src/agents/guardrails.py`.

In [ ]:
# ---------------------------------------------------------------------------
# PII Guardrails — Layer 3: text-level masking and validation
# Layer 1 (SQL) and Layer 2 (DataFrame) are enforced in the tool functions.
# Here we demonstrate the final defense layer.
# ---------------------------------------------------------------------------

from src.agents.guardrails import (
    mask_pii_in_text,
    mask_pii_in_dataframe,
    validate_sql_query,
    validate_agent_output,
    load_guardrail_config,
)

gc = load_guardrail_config(str(PROJECT_ROOT / "configs" / "agents.yaml"))

# --- Demo 1: mask_pii_in_text ---
print("=" * 70)
print("LAYER 3: Text-level PII masking")
print("=" * 70)

test_texts = [
    "Member João Silva, CPF 123.456.789-01, is at high churn risk.",
    "Contact her at maria@email.com or (11) 99876-5432 for follow-up.",
    "Location LOC_0042 has 320 active members. No PII here.",
]

for text in test_texts:
    masked = mask_pii_in_text(text, gc.pii_mask_token)
    changed = "MASKED" if masked != text else "clean"
    print(f"\n  Input:  {text}")
    print(f"  Output: {masked}")
    print(f"  Status: [{changed}]")

In [ ]:
# --- Demo 2: SQL guardrails ---
print("=" * 70)
print("SQL GUARDRAILS: blocking dangerous patterns")
print("=" * 70)

test_queries = [
    ("SELECT location_id, churn_rate FROM gold.location_metrics", "VALID"),
    ("WITH cte AS (SELECT * FROM members) SELECT count(*) FROM cte", "VALID"),
    ("DROP TABLE members; --", "BLOCKED"),
    ("DELETE FROM gold.members WHERE status = 'inactive'", "BLOCKED"),
    ("UPDATE members SET status = 'active' WHERE id = 1", "BLOCKED"),
]

for query, expected in test_queries:
    try:
        validated = validate_sql_query(query, gc.blocked_patterns)
        status = "PASSED"
    except Exception as e:
        status = "BLOCKED"

    icon = "pass" if status == expected else "MISMATCH"
    print(f"\n  Query:    {query[:60]}{'...' if len(query) > 60 else ''}")
    print(f"  Expected: {expected}  |  Got: {status}  [{icon}]")

In [ ]:
# --- Demo 3: validate_agent_output (catches PII that survived upstream) ---
print("=" * 70)
print("OUTPUT VALIDATION: final defense before response reaches user")
print("=" * 70)

import pandas as pd

test_outputs = [
    "Your location has 320 active members with 8.7% churn rate.",
    "Member with CPF 987.654.321-00 has high churn risk. Contact at joao@gym.com.",
    "We recommend SMS re-engagement for 23 at-risk members. Call (21) 3456-7890 for support.",
]

for output in test_outputs:
    validated = validate_agent_output(output, gc.pii_mask_token)
    changed = "PII CAUGHT AND MASKED" if validated != output else "clean"
    print(f"\n  Input:  {output}")
    print(f"  Output: {validated}")
    print(f"  Status: [{changed}]")

# --- Demo 4: DataFrame-level masking ---
print("\n" + "=" * 70)
print("LAYER 2: DataFrame PII masking")
print("=" * 70)

sample_df = pd.DataFrame({
    "member_id": ["MEM_001", "MEM_002", "MEM_003"],
    "member_name": ["João Silva", "Maria Santos", "Pedro Costa"],
    "cpf": ["123.456.789-01", "987.654.321-00", "111.222.333-44"],
    "email": ["joao@test.com", "maria@test.com", "pedro@test.com"],
    "churn_probability": [0.82, 0.45, 0.91],
    "plan_price": [149.90, 119.90, 89.90],
})

print("\nBefore masking:")
print(sample_df.to_string(index=False))

masked_df = mask_pii_in_dataframe(sample_df, gc.pii_fields, gc.pii_mask_token)
print("\nAfter masking:")
print(masked_df.to_string(index=False))
print("\nThree layers of defense. If PII reaches the user, all three failed.")

## 4. Tool System

Each agent has access to **specific tools** — this isn't just access control, it's architectural. Each agent's prompt is optimized for its tool set. When an agent has fewer, more focused tools, the LLM makes better tool selection decisions.

The Analyst can query data but can't run the optimizer. The Strategist can estimate treatment effects but can't write SQL. This separation mirrors how human analytics teams work: the data engineer doesn't make strategy recommendations, and the strategist doesn't debug ETL pipelines.

Tool definitions live in `src/agents/tools.py`. Every tool enforces PII masking and structured logging.

In [ ]:
# ---------------------------------------------------------------------------
# Tool system: list tools per agent from configuration
# ---------------------------------------------------------------------------

import inspect
from src.agents import tools as agent_tools

print("TOOL ASSIGNMENTS PER AGENT")
print("=" * 70)

for agent_name, agent_def in agent_config["agents"].items():
    print(f"\n{agent_name.upper()}: {agent_def['description']}")

    if "tools" in agent_def:
        for tool_name in agent_def["tools"]:
            # Get the function signature from the tools module if available
            tool_fn = getattr(agent_tools, tool_name, None)
            if tool_fn:
                sig = inspect.signature(tool_fn)
                doc_first_line = (tool_fn.__doc__ or "").strip().split("\n")[0]
                print(f"  - {tool_name}{sig}")
                print(f"    {doc_first_line}")
            else:
                print(f"  - {tool_name} (defined in production, not in prototype)")
    elif "rules" in agent_def:
        print("  Tools: none (formatting only)")
        print("  Rules:")
        for rule in agent_def["rules"]:
            print(f"    - {rule}")

## 5. Query Classification

The router classifies queries before routing them to the appropriate agent pipeline. This is intentionally a lightweight step — in the current implementation it uses keyword matching (see `src/agents/graph.py`), though production could use a fine-tuned classifier or a fast LLM call with structured output.

The classification determines the *depth* of processing:
- **Factual** queries need a single data lookup — no causal analysis, no optimization.
- **Diagnostic** queries need data *and* causal reasoning — "why" requires understanding drivers, not just metrics.
- **Prescriptive** queries need the full stack — data, causal effects, and the optimizer to generate actionable recommendations.
- **Conversational** queries skip analysis entirely — the writer handles greetings, clarifications, and out-of-scope requests.

In [ ]:
# ---------------------------------------------------------------------------
# Query classification demo
# Uses the router logic from src/agents/graph.py
# ---------------------------------------------------------------------------

from src.agents.graph import create_router_node

router = create_router_node()

test_queries = [
    ("How many members churned last month?",              "factual"),
    ("What's the current churn rate at my gym?",          "factual"),
    ("Why is churn increasing at location 42?",           "diagnostic"),
    ("What's causing members to leave in their 2nd month?", "diagnostic"),
    ("What should I do to reduce churn at my gym?",       "prescriptive"),
    ("How should I allocate my retention budget?",        "prescriptive"),
    ("Good morning!",                                      "conversational"),
    ("Thanks, that's helpful.",                            "conversational"),
]

print("QUERY CLASSIFICATION")
print("=" * 90)
print(f"{'Query':<55} {'Expected':<16} {'Got':<16} {'Match'}")
print("-" * 90)

for query, expected in test_queries:
    # Create a minimal state to run the router
    state = {
        "messages": [{"role": "user", "content": query}],
        "context": {},
        "tools_called": [],
        "confidence": 0.5,
        "query": query,
        "query_type": "",
        "location_id": None,
        "session_id": "demo",
        "analysis_result": None,
        "strategy_result": None,
        "final_response": "",
        "error": None,
    }

    result = router(state)
    got = result["query_type"]
    match = "yes" if got == expected else "NO"
    print(f"  {query:<53} {expected:<16} {got:<16} {match}")

# Show the routing paths
print("\n\nROUTING PATHS:")
paths = {
    "factual":        "Router -> Analyst -> Writer -> END",
    "diagnostic":     "Router -> Analyst -> Strategist -> Writer -> END",
    "prescriptive":   "Router -> Analyst -> Strategist -> Writer -> END",
    "conversational": "Router -> Writer -> END",
}
for category, path in paths.items():
    print(f"  {category:<16} {path}")

## 6. Memory System

Without memory, every conversation starts from zero. A manager asks about churn at their location today and gets an analysis. Two weeks later they ask again — the agent has no idea it analyzed the same location before, so it can't detect that churn *escalated* from 10% to 15%.

The memory system uses **pgvector** (PostgreSQL with vector similarity search) to provide three types of memory:

- **Conversation memory:** recent messages for context continuity within a session. Windowed to the last 10 messages (configurable) to control token cost.
- **Episodic memory:** past analyses indexed by location and time. Enables trend detection: *"Last time we analyzed Location 42 (2 weeks ago), churn was 10%. Now it's 15% — escalating trend."*
- **Semantic memory:** domain knowledge retrieval via embedding similarity. What is an aggregator member? What seasonal patterns exist in Brazilian fitness? Retrieved from a curated knowledge base, not from the LLM's parametric memory.

The trade-off vs. full context stuffing: memory retrieval may miss relevant context, but it scales to hundreds of locations with months of interaction history. Full context would blow the token budget within a few sessions.

*Implementation: `src/agents/memory.py` — `AgentMemoryStore` with store, search, and session management.*

In [ ]:
# ---------------------------------------------------------------------------
# Memory system: demonstrate the AgentMemoryStore interface
# Note: actual pgvector operations require a running PostgreSQL instance.
# Here we show the interface and explain the design.
# ---------------------------------------------------------------------------

from src.agents.memory import AgentMemoryStore

# Show the interface — the store, search, and retrieve methods
print("AgentMemoryStore Interface")
print("=" * 70)

methods = [
    ("store_memory", "Store a memory with embedding for later retrieval"),
    ("search_similar", "Find memories similar to a query (cosine similarity)"),
    ("get_conversation_history", "Retrieve recent messages for a session"),
    ("clear_session", "Delete all memories for a session (LGPD compliance)"),
]

for method_name, description in methods:
    method = getattr(AgentMemoryStore, method_name)
    sig = inspect.signature(method)
    print(f"\n  {method_name}{sig}")
    print(f"    {description}")
    # Show first line of docstring for more detail
    doc_lines = (method.__doc__ or "").strip().split("\n")
    if doc_lines:
        print(f"    Detail: {doc_lines[0].strip()}")

print("\n" + "=" * 70)
print("MEMORY CONFIGURATION (from configs/agents.yaml)")
print("=" * 70)
mem_config = agent_config["memory"]
print(f"  Backend:              {mem_config['backend']}")
print(f"  Conversation window:  {mem_config['conversation_window']} messages")
print(f"  Long-term retrieval:  top-{mem_config['long_term_top_k']} results")
print(f"  Similarity threshold: {mem_config['similarity_threshold']} (cosine)")
print(f"\n  Embedding: {agent_config['embedding']['model']} ({agent_config['embedding']['dimensions']} dims)")

# Simulated memory interaction flow
print("\n" + "=" * 70)
print("SIMULATED MEMORY FLOW")
print("=" * 70)
print("""
  1. Manager asks: "Why is churn up at my gym?"
     → Agent processes query, generates analysis
     → store_memory(content="LOC_0042 analysis: churn 15%, up from 10%",
                     metadata={"session_id": "abc", "location_id": "LOC_0042",
                               "query_type": "diagnostic"})

  2. Two weeks later, manager asks: "How is churn trending?"
     → search_similar("churn trend LOC_0042", memory_type="semantic")
     → Returns: "LOC_0042 analysis (2 weeks ago): churn was 10%"
     → Agent compares: 10% → 15% = escalating trend
     → Response: "Churn at your location increased from 10% to 15%
                  over the past 2 weeks — this is an escalating trend."

  3. Manager requests data deletion (LGPD right):
     → clear_session("abc")
     → All memories for session "abc" permanently deleted
""")

## 7. Agent Evaluation Framework

The hardest part of production agents isn't building them — it's knowing when they're wrong. Our evaluation framework tests five dimensions:

1. **Task success rate:** correct answer for 50+ canonical queries. Each scenario specifies expected content substrings — if the response contains them, it passes.
2. **Tool selection accuracy:** did the agent call the right tools? A factual query should trigger `query_location_metrics`, not `run_optimizer`.
3. **Factual grounding:** every number must trace to a tool call result. The agent should never cite statistics that didn't come from a tool.
4. **PII leak rate:** zero tolerance. Any PII in any response is a critical failure, regardless of whether other metrics look good.
5. **Cost efficiency:** tokens per query, tracked over time. An agent that burns 10K tokens per simple lookup is too expensive at scale.

Without this, you're deploying a system you can't measure. The evaluation runs in CI on every PR that touches agent code.

*Implementation: `src/agents/eval/evaluator.py` — `EvalScenario`, `evaluate_agent()`, `check_pii_leakage()`.*

In [ ]:
# ---------------------------------------------------------------------------
# Evaluation framework: scenario structure and example scenarios
# ---------------------------------------------------------------------------

from src.agents.eval import EvalScenario, ScenarioResult, check_pii_leakage

print("EvalScenario structure:")
print(f"  {inspect.signature(EvalScenario.__init__)}")
print()

# Example evaluation scenarios — these would live in configs/eval_scenarios.yaml
example_scenarios = [
    EvalScenario(
        query="How many members churned last month at location 42?",
        expected_tools=["query_location_metrics"],
        expected_answer_contains=["members", "churn"],
        category="factual",
        location_id="LOC_0042",
        description="Basic churn count lookup",
    ),
    EvalScenario(
        query="Why is churn increasing at location 42?",
        expected_tools=["query_location_metrics", "compare_cohorts"],
        expected_answer_contains=["churn", "increase"],
        category="diagnostic",
        location_id="LOC_0042",
        description="Churn root cause analysis",
    ),
    EvalScenario(
        query="What should I do to reduce churn at my gym?",
        expected_tools=["query_location_metrics", "estimate_treatment_effect", "run_optimizer"],
        expected_answer_contains=["recommend", "members"],
        category="prescriptive",
        location_id="LOC_0042",
        description="Prescriptive retention recommendation",
    ),
    EvalScenario(
        query="Good morning, how are you?",
        expected_tools=[],
        expected_answer_contains=[],
        category="conversational",
        description="Greeting — should not trigger analysis tools",
    ),
]

print("EXAMPLE EVALUATION SCENARIOS")
print("=" * 90)
for i, scenario in enumerate(example_scenarios, 1):
    print(f"\n  Scenario {i}: {scenario.description}")
    print(f"    Category:        {scenario.category}")
    print(f"    Query:           {scenario.query}")
    print(f"    Expected tools:  {scenario.expected_tools}")
    print(f"    Must contain:    {scenario.expected_answer_contains}")

# PII leak detection demo
print("\n\n" + "=" * 90)
print("PII LEAK DETECTION (zero tolerance)")
print("=" * 90)
pii_test_cases = [
    ("Your location has 320 members with 8.7% churn.", False),
    ("Member CPF 123.456.789-01 is at risk.", True),
    ("Contact joao@gym.com for details.", True),
    ("Recommend SMS for 23 members at LOC_0042.", False),
]

for response, expected_leak in pii_test_cases:
    leaked = check_pii_leakage(response)
    status = "LEAK DETECTED" if leaked else "clean"
    match = "correct" if leaked == expected_leak else "WRONG"
    print(f"  [{status:>14}] ({match:>7}) {response[:60]}")

In [ ]:
# ---------------------------------------------------------------------------
# How evaluation runs in CI (conceptual — actual CI config in .github/workflows/)
# ---------------------------------------------------------------------------

print("EVALUATION IN CI")
print("=" * 70)
print("""
The evaluation pipeline runs on every PR that touches src/agents/:

  1. Load scenarios from configs/eval_scenarios.yaml
  2. Run each scenario through the agent (with mocked LLM for determinism)
  3. Check metrics against thresholds from configs/agents.yaml:

     Metric                    Threshold    Action on failure
     ─────────────────────────────────────────────────────────
     task_success_rate         >= 0.70      Block merge
     pii_leak_rate             == 0.00      Block merge + alert
     avg_latency_seconds       <= 15.0      Warning only
     tool_selection_accuracy   >= 0.70      Block merge

  4. Report results as PR comment via GitHub Actions
  5. Track metrics over time in MLflow for drift detection

Key design choice: we mock the LLM in CI tests (deterministic responses)
but run live LLM evaluation weekly on a staging environment. This catches
both code regressions (CI) and model behavior changes (weekly).
""")

# Show the evaluation thresholds from config
print("Configured thresholds:")
for metric, threshold in agent_config["evaluation"]["thresholds"].items():
    severity = "CRITICAL" if "pii" in metric else "standard"
    print(f"  {metric:<30} {threshold:<10} [{severity}]")

## 8. Simulated Agent Interaction

Let's simulate what a full conversation looks like end-to-end. In production, this runs through LangGraph with actual LLM calls (see `src/agents/graph.py`). Here we trace the routing, tool calls, and output formatting to show how the pieces connect.

The diagnostic query "Why is churn up at location 42?" triggers the full Analyst-Strategist-Writer pipeline — the most complex path through the graph.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated agent interaction: trace a diagnostic query through the pipeline
# In production, this runs via src.agents.graph.run_agent()
# ---------------------------------------------------------------------------

from src.agents.tools import (
    query_location_metrics,
    compare_cohorts,
    estimate_treatment_effect,
)

print("=" * 80)
print("SIMULATED AGENT INTERACTION")
print("=" * 80)

# Step 1: User query
query = "Why is churn up at location 42?"
print(f'\nManager: "{query}"')
print()

# Step 2: Router classifies
state = {
    "messages": [{"role": "user", "content": query}],
    "context": {}, "tools_called": [], "confidence": 0.5,
    "query": query, "query_type": "", "location_id": "LOC_0042",
    "session_id": "demo_001", "analysis_result": None,
    "strategy_result": None, "final_response": "", "error": None,
}
router_result = router(state)
print(f"Router: {router_result['query_type']} -> Analyst -> Strategist -> Writer")
print()

# Step 3: Analyst executes tools
print("Analyst calls:")
metrics = query_location_metrics("LOC_0042", "last_30d")
print(f'  query_location_metrics("LOC_0042", "last_30d")')
print(f'    -> churn_rate={metrics["churn_rate"]}, active={metrics["active_members"]}, '
      f'at_risk={metrics["at_risk_members"]}')

prior_metrics = query_location_metrics("LOC_0042", "prior_30d")
print(f'  query_location_metrics("LOC_0042", "prior_30d")')

cohort_result = compare_cohorts(
    {"location": "LOC_0042", "period": "last_30d"},
    {"location": "LOC_0042", "period": "prior_30d"},
)
print(f'  compare_cohorts(current_period, prior_period)')
print(f'    -> significant differences: {cohort_result["significant_differences"]}')
print()

# Step 4: Strategist estimates treatment effects
print("Strategist calls:")
cate = estimate_treatment_effect("regular_month2-5", "sms_reengagement")
print(f'  estimate_treatment_effect("regular_month2-5", "sms_reengagement")')
print(f'    -> CATE={cate["cate_mean"]:.2f} (CI: {cate["ci_lower"]:.2f}-{cate["ci_upper"]:.2f}), '
      f'n={cate["n_members_in_segment"]}')
print()

# Step 5: Writer formats response
print("Writer formats response:")
print("-" * 70)

# Simulate the writer's output (in production, this is generated by the LLM
# with strict formatting rules from configs/agents.yaml)
response = """Churn at your location increased from 8% to 12% this month.
The main driver is a drop in visit frequency among members in their
2nd-5th month. We recommend prioritizing SMS re-engagement for these
23 members — based on our analysis, this could retain approximately
4-5 of them, representing about R$2,800/month in recurring revenue.

Action items:
  1. SMS re-engagement campaign for 23 at-risk members (automated, R$57.50)
  2. Follow-up phone calls for the 8 highest-value members (R$120.00)
  3. Review next month to check if the trend reverses"""

# Final PII check
validated = validate_agent_output(response, gc.pii_mask_token)
print(f"\n{validated}")
print("-" * 70)
print(f"\nPII check: {'clean' if validated == response else 'MASKED'}")
print(f"Tools used: query_location_metrics, compare_cohorts, estimate_treatment_effect")
print(f"Query type: diagnostic")
print(f"Confidence: 0.82")

In [ ]:
# ---------------------------------------------------------------------------
# Additional query type examples — showing the different pipeline depths
# ---------------------------------------------------------------------------

print("=" * 80)
print("QUERY TYPE EXAMPLES: different paths through the agent graph")
print("=" * 80)

examples = [
    {
        "query": "How many members churned last month?",
        "type": "factual",
        "path": "Router -> Analyst -> Writer -> END",
        "tools": ["query_location_metrics"],
        "response": (
            "Your location lost 28 members last month (8.8% churn rate). "
            "That's 320 active members remaining, with 45 currently at risk."
        ),
    },
    {
        "query": "Why is churn increasing at location 42?",
        "type": "diagnostic",
        "path": "Router -> Analyst -> Strategist -> Writer -> END",
        "tools": ["query_location_metrics", "compare_cohorts", "estimate_treatment_effect"],
        "response": (
            "Churn at your location increased from 8% to 12% this month. "
            "The main driver is a drop in visit frequency among members in "
            "their 2nd-5th month. SMS re-engagement for these 23 members "
            "could retain 4-5 of them (~R$2,800/month)."
        ),
    },
    {
        "query": "What should I do to reduce churn at my gym?",
        "type": "prescriptive",
        "path": "Router -> Analyst -> Strategist -> Writer -> END",
        "tools": [
            "query_location_metrics", "estimate_treatment_effect",
            "run_optimizer",
        ],
        "response": (
            "Based on your budget and member mix, here's your action plan: "
            "1) SMS re-engagement for 20 members (R$50), "
            "2) Phone calls for 15 high-value members (R$225), "
            "3) Discount offers for 7 regulars at risk (R$350). "
            "Expected ROI: 3.2x. Total cost: R$625."
        ),
    },
    {
        "query": "Good morning!",
        "type": "conversational",
        "path": "Router -> Writer -> END",
        "tools": [],
        "response": (
            "Good morning! I can help you with retention analysis for your "
            "location. Try asking about current churn metrics, specific "
            "member risk scores, or recommended retention actions."
        ),
    },
]

for ex in examples:
    print(f'\n{"─" * 80}')
    print(f'Manager: "{ex["query"]}"')
    print(f'  Type:     {ex["type"]}')
    print(f'  Path:     {ex["path"]}')
    print(f'  Tools:    {ex["tools"]}')
    print(f'  Response: {ex["response"]}')

## 9. Production Considerations

What this prototype doesn't show but production requires:

| Concern | Implementation | Status |
|---|---|---|
| **Rate limiting** | 30 queries/user/hour to control LLM cost. Tracked per session in Redis. | Planned (v1.1) |
| **Fallback chain** | LLM timeout (10s) -> cached response -> "I can't answer right now, please try again." | Implemented in `src/agents/graph.py` via confidence check |
| **A/B testing** | Compare agent recommendations vs manager intuition. Track which members were actually retained. | Planned (v2) |
| **Cost tracking** | Token usage per query, per user, per location. Logged via audit system. | Implemented in `configs/agents.yaml` audit config |
| **Latency budget** | 15s p95 target. If tool calls exceed 10s, return partial results with a note. | Threshold configured, partial response logic planned |
| **Human-in-the-loop** | For recommendations above R\$500, require manager confirmation before execution. | Planned (v1.1) |
| **Conversation handoff** | If confidence drops below 0.4 on two consecutive queries, offer to connect with analytics team. | Planned (v1.1) |
| **Multi-language** | Agent responds in Brazilian Portuguese by default (detected from user query language). | Planned (v1.2) |

The biggest risk is not technical — it's **trust calibration**. If the agent gives a confident wrong answer about retention strategy and a manager acts on it, we lose credibility for the entire system. The confidence threshold (0.6, configured in `configs/agents.yaml`) and explicit "I don't know" fallback are the primary defenses against this.

In [ ]:
# ---------------------------------------------------------------------------
# Architecture summary: component inventory and their locations
# ---------------------------------------------------------------------------

print("RETENTIONIQ AGENT SYSTEM — COMPONENT MAP")
print("=" * 80)

components = [
    ("src/agents/graph.py",        "LangGraph state machine",       "Router, Analyst, Strategist, Writer nodes"),
    ("src/agents/tools.py",        "Agent tool functions",           "6 tools with PII masking and audit logging"),
    ("src/agents/guardrails.py",   "PII masking + SQL validation",  "3-layer defense, CPF/email/phone regex"),
    ("src/agents/memory.py",       "pgvector memory store",         "Conversation + semantic memory"),
    ("src/agents/eval/",           "Evaluation framework",          "Scenario-based testing, CI integration"),
    ("configs/agents.yaml",        "Agent configuration",           "LLM params, tools, guardrails, thresholds"),
    ("configs/optimization.yaml",  "Optimizer configuration",       "Budget, actions, stochastic params"),
    ("src/optimization/",          "Budget optimizer",              "Deterministic + stochastic MILP"),
    ("src/api/",                   "FastAPI endpoints",             "REST API for agent invocation"),
]

print(f"\n{'File':<35} {'Component':<30} {'Description'}")
print("-" * 100)
for path, component, desc in components:
    print(f"  {path:<33} {component:<30} {desc}")

print(f"\n\nData flow:")
print("""
  User query (natural language)
    -> FastAPI endpoint (src/api/)
      -> PII masking on input (guardrails)
        -> LangGraph state machine (graph.py)
          -> Router classifies query
            -> Analyst queries data (tools.py)
              -> Strategist runs causal + optimizer (tools.py)
                -> Writer formats response
                  -> PII validation on output (guardrails)
                    -> Response to user
                      -> Audit log (structured logging)
                        -> Memory store (pgvector)
""")

## 10. Key Takeaways

1. **Agents make the entire analytical stack accessible to non-technical users.** A location manager asks a question in natural language and gets an answer that draws from data queries, survival models, causal estimates, and budget optimization — without knowing any of those systems exist.

2. **Three-layer PII defense ensures LGPD compliance at the code level.** SQL column exclusion, DataFrame masking, and output regex scanning form a defense-in-depth strategy. A single layer failing doesn't expose PII.

3. **Specialized agents > one general agent for tool selection accuracy.** The router-analyst-strategist-writer pattern keeps each agent focused on a small tool set. This improves tool selection reliability and makes prompts more maintainable than a single monolithic agent with 15+ tools.

4. **Evaluation framework is non-negotiable — you can't deploy what you can't measure.** Scenario-based evaluation with automated CI checks ensures that agent quality doesn't degrade silently. Zero tolerance for PII leaks means a single failure blocks deployment.

5. **Memory enables trend detection across conversations, not just single-shot Q&A.** The pgvector-backed memory store allows the agent to compare current metrics against past analyses, surfacing escalating trends that a memoryless system would miss.

**What this prototype defers to production:**
- Actual LLM integration (LangGraph + Anthropic API) — prototyped here with deterministic mock responses
- Real pgvector memory operations — demonstrated via interface, requires running PostgreSQL
- Rate limiting, cost tracking, and human-in-the-loop workflows — architecture designed, implementation in v1.1
- A/B testing of agent recommendations vs manager intuition — planned for v2

---

*Previous notebook: [05_optimization.ipynb](./05_optimization.ipynb) — budget optimization under uncertainty*  
*Production code: `src/agents/` — LangGraph graph, tools, guardrails, memory, evaluation*  
*Architecture decisions: `docs/ARCHITECTURE.md`*